In [1]:
import pandas as pd
import numpy as np
import math
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
from datetime import datetime
import warnings
import os
from scipy import stats
import pickle
from dateutil.relativedelta import relativedelta
warnings.filterwarnings('ignore')

In [2]:
# 0 -- 6 month
# 1 -- 2 year
# 2 -- 5 year
# 3 -- 10 year
rate_reg = 3

In [3]:
load_path = r'\\osiride-fs\group\main\891af\private\Area_MF\Varie\Poli_Venturi\Data\Database_Regression.xlsx'
mergedData = pd.read_excel(load_path,index_col=[0],parse_dates=[0])
mergedData = mergedData.loc[datetime(2000,1,1):datetime(2025,1,1),[
    'SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd',
    'SwapESTR6Mm','SwapESTR2Ym','SwapESTR5Ym','SwapESTR10Ym',
    'OASd','EASurprise','USSurprise'
]]
mergedData.dropna(inplace=True)
    
diff_rates_clean = mergedData.loc[:,['SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd']]

z = np.abs(stats.zscore(diff_rates_clean))
threshold = 10

outliers = pd.concat([diff_rates_clean[z.SwapESTR6Md>threshold],diff_rates_clean[z.SwapESTR2Yd>threshold],
                     diff_rates_clean[z.SwapESTR5Yd>threshold],diff_rates_clean[z.SwapESTR10Yd>threshold],
                     diff_rates_clean.loc[datetime(2023,3,13):datetime(2023,3,15)]],axis=0)

outliers.reset_index(inplace=True,drop=False)
outliers.drop_duplicates(subset='Date',inplace=True)
outliers.sort_values(by='Date',inplace=True,ignore_index=True)
outliers.set_index('Date',inplace=True,drop=True)

diff_rates_clean.drop(outliers.index,inplace=True,axis=0)

idx = diff_rates_clean.index

mergedData = mergedData.loc[idx]

In [4]:
ratevol_estr = mergedData.loc[:,['SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd']].fillna(method='ffill').rolling(window=5,closed='left').std()
ratevol_estr = (ratevol_estr - ratevol_estr.mean())/ratevol_estr.std()
for j in ratevol_estr.columns:
    ratevol_estr.rename(columns={j: j.replace('Yd','Yv')},inplace=True)
    ratevol_estr.rename(columns={j: j.replace('Md','Mv')},inplace=True)

diff_rates_estr = mergedData.loc[:,['SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd']]

df_momentum_estr = mergedData.loc[:,['SwapESTR6Mm','SwapESTR2Ym','SwapESTR5Ym','SwapESTR10Ym']]
df_momentum_estr.fillna(method='ffill',inplace=True)
df_momentum_estr = (df_momentum_estr - df_momentum_estr.mean())/df_momentum_estr.std()
for j in df_momentum_estr.columns:
    df_momentum_estr.rename(columns={j: j.replace('Y','Ym')},inplace=True)
    df_momentum_estr.rename(columns={j: j.replace('M','Mm')},inplace=True)

diff_oas = mergedData.loc[:,['OASd']]
diff_oas = (diff_oas - diff_oas.mean())/diff_oas.std()

df_surprise_ea = mergedData.loc[:,['EASurprise']]
df_surprise_ea = (df_surprise_ea - df_surprise_ea.mean())/df_surprise_ea.std()

df_surprise_us = mergedData.loc[:,['USSurprise']]
df_surprise_us = (df_surprise_us - df_surprise_us.mean())/df_surprise_us.std()

In [5]:
df_reg = pd.concat([diff_rates_estr.iloc[:,rate_reg],
                    df_surprise_ea,df_surprise_us,
                    df_momentum_estr.iloc[:,rate_reg],ratevol_estr.iloc[:,rate_reg],diff_oas],axis=1)
    
Y = df_reg.iloc[:,[0]]
X = df_reg.iloc[:,1:]
X = sm.add_constant(X)

model = sm.OLS(Y, X, missing='drop').fit(cov_type='HAC',cov_kwds={'maxlags':math.ceil(0.75*len(Y)**(1/3))})
display(model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           SwapESTR10Yd   R-squared:                       0.163
Model:                            OLS   Adj. R-squared:                  0.163
Method:                 Least Squares   F-statistic:                     172.8
Date:                Sun, 08 Feb 2026   Prob (F-statistic):          1.26e-172
Time:                        12:22:27   Log-Likelihood:                -17154.
No. Observations:                6229   AIC:                         3.432e+04
Df Residuals:                    6223   BIC:                         3.436e+04
Df Model:                           5                                         
Covariance Type:                  HAC                                         
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.0545      0.036     -1.496      0.135      -0.126       0.017
EASurprise        0.2880      0.057      5.016      0.000       0.175       0.400
USSurprise        0.1498      0.050      2.999      0.003       0.052       0.248
SwapESTR10Ymm     1.1930      0.052     22.785      0.000       1.090       1.296
SwapESTR10Yv      0.1085      0.069      1.565      0.117      -0.027       0.244
OASd             -1.0049      0.137     -7.339      0.000      -1.273      -0.737
==============================================================================
Omnibus:                      699.113   Durbin-Watson:                   1.998
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             6700.781
Skew:                           0.030   Prob(JB):                         0.00
Kurtosis:                       8.081   Cond. No.                         1.16
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 14 lags and without small sample correction
"""